# 04 — Visualizacion de resultados

In [3]:
exec(open('/home/alumno/Desktop/bigdata-nyc-taxi/export_streaming_json.py').read())

HTML autocontenido generado: /home/alumno/Desktop/bigdata-nyc-taxi/nyc_taxi_viz_standalone.html


In [4]:
import json
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

STREAMING_OUTPUT = "/home/alumno/Desktop/bigdata-nyc-taxi/streaming_output/"
HTML_TEMPLATE = "/home/alumno/Desktop/bigdata-nyc-taxi/nyc_taxi_viz.html"
OUTPUT_HTML = "/home/alumno/Desktop/bigdata-nyc-taxi/nyc_taxi_viz_standalone.html"

spark = SparkSession.builder.master("local[*]").appName("export-viz").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df = spark.read.parquet(STREAMING_OUTPUT)
df = df.withColumn("zone_lon", F.round("zone_lon", 2)) \
       .withColumn("zone_lat", F.round("zone_lat", 2)) \
       .withColumn("window_start", F.date_format("window_start", "yyyy-MM-dd HH:mm:ss"))

rows = df.orderBy("window_start", "zone_lon", "zone_lat").collect()

windows = {}
for row in rows:
    ts = row["window_start"]
    if ts not in windows:
        windows[ts] = []
    windows[ts].append({
        "lon": row["zone_lon"],
        "lat": row["zone_lat"],
        "real": round(float(row["trip_count"]), 1),
        "pred": round(float(row["prediction"]), 1),
    })

mae_by_window = {}
for ts, zones in windows.items():
    errors = [abs(z["real"] - z["pred"]) for z in zones]
    mae_by_window[ts] = round(sum(errors) / len(errors), 2)

result = {
    "timestamps": sorted(windows.keys()),
    "windows": windows,
    "mae": mae_by_window,
}

json_str = json.dumps(result, separators=(",", ":"))

with open(HTML_TEMPLATE, "r") as f:
    html = f.read()

# Reemplazar el bloque fetch completo por datos inline
fetch_block = """fetch('streaming_viz.json')
  .then(r => { if (!r.ok) throw new Error('not found'); return r.json(); })"""

inline_block = f"Promise.resolve({json_str})\n  .then(d => d)"

html = html.replace(fetch_block, inline_block)

with open(OUTPUT_HTML, "w") as f:
    f.write(html)

print(f"Listo: {OUTPUT_HTML}")
spark.stop()

Listo: /home/alumno/Desktop/bigdata-nyc-taxi/nyc_taxi_viz_standalone.html
